In [1]:
# ==============================================================================
# [CELL 1] TẢI MÃ NGUỒN VÀ CÀI ĐẶT TẤT CẢ THƯ VIỆN CẦN THIẾT
# ==============================================================================
import os

%cd /kaggle/working
if not os.path.exists("/kaggle/working/AIC2026"):
    !git clone https://github.com/Thanhdat3010/AIC2026.git

%cd /kaggle/working/AIC2026
!git pull origin master
!pip install -r requirements.txt

print("✅ CELL 1 HOÀN TẤT: Mã nguồn và toàn bộ thư viện đã sẵn sàng!")


/kaggle/working
Cloning into 'AIC2026'...
remote: Enumerating objects: 400, done.
remote: Counting objects: 100% (400/400), done.
remote: Compressing objects: 100% (273/273), done.
remote: Total 400 (delta 207), reused 305 (delta 112), pack-reused 0 (from 0)
Receiving objects: 100% (400/400), 159.97 KiB | 3.72 MiB/s, done.
Resolving deltas: 100% (207/207), done.
/kaggle/working/AIC2026
From https://github.com/Thanhdat3010/AIC2026
 * branch            master     -> FETCH_HEAD
Already up to date.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 108.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━

In [2]:
# ==============================================================================
# [CELL 2] SANITY CHECK: TEST SIGLIP 2 TRÊN GPU & DỌN SẠCH VRAM SAU KHI TEST
# ==============================================================================
import os, sys, gc

# Vá lỗi tương thích Pillow 11 trên Python 3.12
try:
    import PIL._util
    if not hasattr(PIL._util, "is_directory"):
        PIL._util.is_directory = lambda f: isinstance(f, (str, bytes, os.PathLike)) and os.path.isdir(f)
    if not hasattr(PIL._util, "is_path"):
        PIL._util.is_path = lambda f: isinstance(f, (str, bytes, os.PathLike))
except Exception:
    pass

import torch
from PIL import Image
import numpy as np
from transformers import AutoImageProcessor, AutoModel

print("🔍 Đang kiểm tra tải model SigLIP 2 trên GPU T4...")
ckpt = "google/siglip2-so400m-patch14-384"
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoImageProcessor.from_pretrained(ckpt, trust_remote_code=True)
model = AutoModel.from_pretrained(ckpt, trust_remote_code=True).to(device).eval()

# Test thử 1 ảnh giả lập với Mixed Precision float16
dummy_img = Image.fromarray(np.uint8(np.random.rand(384, 384, 3) * 255))
inputs = processor(images=[dummy_img], return_tensors="pt").to(device)

with torch.inference_mode():
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        if hasattr(model, "get_image_features"):
            feats = model.get_image_features(**inputs)
        else:
            outputs = model.vision_model(**inputs)
            feats = outputs.pooler_output if outputs.pooler_output is not None else outputs.last_hidden_state[:, 0]
        feats = feats / feats.norm(dim=-1, keepdim=True)

print("=" * 60)
print(f"🎉 SANITY CHECK THÀNH CÔNG RỰC RỠ!")
print(f"👉 Shape vector: {feats.shape} (Đúng chuẩn 1152 chiều SigLIP 2)")
print(f"👉 Thiết bị chạy: {device} ({torch.cuda.get_device_name(0)})")
print("=" * 60)

# DỌN SẠCH 100% VRAM ĐỂ CELL 3 CÓ TOÀN BỘ BỘ NHỚ TRỐNG
del model, processor, inputs, feats
gc.collect()
torch.cuda.empty_cache()
print("🧹 Đã dọn sạch VRAM GPU, sẵn sàng 100% để chạy Cell 3!")


🔍 Đang kiểm tra tải model SigLIP 2 trên GPU T4...


preprocessor_config.json:   0%|          | 0.00/394 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/559 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

🎉 SANITY CHECK THÀNH CÔNG RỰC RỠ!
👉 Shape vector: torch.Size([1, 1152]) (Đúng chuẩn 1152 chiều SigLIP 2)
👉 Thiết bị chạy: cuda (Tesla T4)
🧹 Đã dọn sạch VRAM GPU, sẵn sàng 100% để chạy Cell 3!


In [3]:
# ==============================================================================
# [CELL 3] CHẠY TRÍCH XUẤT TOÀN BỘ 177.321 KEYFRAMES BẰNG SIGLIP 2
# ==============================================================================
import subprocess
import os
import sys
import glob

%cd /kaggle/working/AIC2026

# Tự động phát hiện vị trí file frames.parquet trong Dataset Kaggle
found_frames = glob.glob('/kaggle/input/**/frames.parquet', recursive=True)
if found_frames:
    frames_path = found_frames[0]
    print(f"🎯 Đã tìm thấy frames.parquet tại: {frames_path}")
else:
    frames_path = "/kaggle/working/AIC2026/data/batch_1/processed/frames.parquet"
    print(f"⚠️ Dùng đường dẫn mặc định: {frames_path}")

output_siglip = "/kaggle/working/siglip_features.npy"

env_vis = os.environ.copy()
env_vis["CUDA_VISIBLE_DEVICES"] = "0"
env_vis["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

cmd_vis = [
    sys.executable, "/kaggle/working/AIC2026/scripts/extract_visual_features.py",
    "--urls_file", "config/drive_keyframes_urls.txt",
    "--frames_path", frames_path,
    "--output_path", output_siglip,
    "--model_name", "google/siglip2-so400m-patch14-384",
    "--batch_size", "32",
    "--device", "cuda"
]

print("🚀 BẮT ĐẦU TRÍCH XUẤT GOOGLE SIGLIP 2 CHO TOÀN BỘ 14 GÓI KEYFRAMES...")
p_vis = subprocess.Popen(cmd_vis, cwd="/kaggle/working/AIC2026", env=env_vis)
ret = p_vis.wait()

if ret == 0:
    print("=" * 60)
    print("🎉🎉🎉 HOÀN TẤT 100%! FILE KẾT QUẢ ĐÃ LƯU TẠI:")
    print(f"📁 {output_siglip}")
    print("=" * 60)
else:
    print(f"❌ Có lỗi xảy ra (mã lỗi: {ret}). Vui lòng xem thông báo bên trên!")


/kaggle/working/AIC2026
🎯 Đã tìm thấy frames.parquet tại: /kaggle/input/datasets/truongthanhdat3010/aic2026-data/frames.parquet
🚀 BẮT ĐẦU TRÍCH XUẤT GOOGLE SIGLIP 2 CHO TOÀN BỘ 14 GÓI KEYFRAMES...
✅ Đã nạp mapping 177,321 keyframes từ: /kaggle/input/datasets/truongthanhdat3010/aic2026-data/frames.parquet
=== Khởi tạo Mô hình Thị giác SOTA SigLIP 2: google/siglip2-so400m-patch14-384 trên cuda ===
✅ Mô hình đã sẵn sàng! Embedding dimension: 1152
📦 File vector đích: /kaggle/working/siglip_features.npy [Shape: (177321, 1152), dtype=float16]
📋 Đã nạp 14 link từ config/drive_keyframes_urls.txt


📦 [TỔNG] Keyframe Packages:   0%|          | 0/14 [00:00<?, ?gói/s, gói_hiện_tại=Keyframes_L21.zip]
📥 [Tải Keyframes 1/14] Keyframes_L21.zip (Lần 1):   0%|          | 0.00/1.35G [00:00<?, ?B/s]
📥 [Tải Keyframes 1/14] Keyframes_L21.zip (Lần 1):   0%|          | 3.00M/1.35G [00:00<00:57, 25.2MB/s]
📥 [Tải Keyframes 1/14] Keyframes_L21.zip (Lần 1):   0%|          | 6.00M/1.35G [00:00<00:56, 25.4MB/s]
📥 [Tải Keyframes 1/14] Keyframes_L21.zip (Lần 1):   1%|          | 9.00M/1.35G [00:00<00:56, 25.5MB/s]
📥 [Tải Keyframes 1/14] Keyframes_L21.zip (Lần 1):   1%|          | 13.0M/1.35G [00:00<00:48, 29.8MB/s]
📥 [Tải Keyframes 1/14] Keyframes_L21.zip (Lần 1):   1%|▏         | 20.0M/1.35G [00:00<00:33, 42.6MB/s]
📥 [Tải Keyframes 1/14] Keyframes_L21.zip (Lần 1):   2%|▏         | 26.0M/1.35G [00:00<00:29, 48.6MB/s]
📥 [Tải Keyframes 1/14] Keyframes_L21.zip (Lần 1):   3%|▎         | 36.0M/1.35G [00:00<00:21, 65.5MB/s]
📥 [Tải Keyframes 1/14] Keyframes_L21.zip (Lần 1):   4%|▎         | 49.0M/1.35G [00:00


⚠️ Mạng gián đoạn khi tải Keyframes_L27.zip (HTTPSConnectionPool(host='aic-data.ledo.io.vn', port=443): Read timed out.). Đang thử lại lần 2/5 sau 3 giây...



📥 [Tải Keyframes 11/14] Keyframes_L27.zip (Lần 2):   0%|          | 0.00/1.02G [00:00<?, ?B/s]
📥 [Tải Keyframes 11/14] Keyframes_L27.zip (Lần 2):   0%|          | 1.00M/1.02G [04:10<72:12:35, 4.19kB/s]
📥 [Tải Keyframes 11/14] Keyframes_L27.zip (Lần 2):   0%|          | 1.00M/1.02G [04:29<72:12:35, 4.19kB/s]
📥 [Tải Keyframes 11/14] Keyframes_L27.zip (Lần 2):   0%|          | 2.00M/1.02G [06:36<54:30:18, 5.55kB/s]
📥 [Tải Keyframes 11/14] Keyframes_L27.zip (Lần 2):   0%|          | 3.00M/1.02G [06:37<29:47:08, 10.1kB/s]
📥 [Tải Keyframes 11/14] Keyframes_L27.zip (Lần 2):   0%|          | 4.00M/1.02G [06:38<18:05:51, 16.7kB/s]
📥 [Tải Keyframes 11/14] Keyframes_L27.zip (Lần 2):   0%|          | 5.00M/1.02G [06:39<11:37:34, 25.9kB/s]
📥 [Tải Keyframes 11/14] Keyframes_L27.zip (Lần 2):   1%|          | 6.00M/1.02G [06:39<7:42:43, 39.1kB/s] 
📥 [Tải Keyframes 11/14] Keyframes_L27.zip (Lần 2):   1%|          | 7.00M/1.02G [06:40<5:13:35, 57.6kB/s]
📥 [Tải Keyframes 11/14] Keyframes_L27.zip (Lần 2)


🎉 HOÀN TẤT TRÍCH XUẤT ĐẶC TRƯNG THỊ GIÁC SOTA: /kaggle/working/siglip_features.npy
⏱️ Tổng thời gian: 18555s (5.15 giờ)
🎉🎉🎉 HOÀN TẤT 100%! FILE KẾT QUẢ ĐÃ LƯU TẠI:
📁 /kaggle/working/siglip_features.npy
